In [22]:
from itertools import combinations

# Definimos el cuerpo cuadrático imaginario.
K.<a> = QuadraticField(-21)

R.<x> = K[]

# Anillo de enteros de K.
OK = K.ring_of_integers()

# Grupo de clases y número de clase de K.
Cl = OK.class_group()
h = Cl.order()

# Construimos una extensión de K adjuntando las raíces indicadas.
L1.<b> = K.extension(x^2 - 2)
L2.<c> = L1.extension(x^2 + a + 2)

# Calculamos un cuerpo que contiene a L1 y L2.
L = L1.composite_fields(L2)[0]

print(L.base_field())

Number Field in b with defining polynomial x^2 - 2 over its base field


In [23]:
def grupos_irreducibles(datos, n):
    """
    Encuentra los grupos de ideales que corresponden a una
    partición de los n bits, sin repetir elementos ni considerar
    distintos los grupos que solo difieren en el orden.

    El resultado contiene únicamente los ideales.
    """

    objetivo = 2^n - 1
    k = len(datos)

    resultados = []
    vistos = set()

    def bt(indice, mascara, grupo):
        if mascara == objetivo:
            ideales = [par[1] for par in grupo]
            clave = tuple(sorted(ideales))

            if clave not in vistos:
                vistos.add(clave)
                resultados.append(ideales)

            return

        for j in range(indice, k):
            a, I = datos[j]

            if mascara & a:
                continue

            nueva = mascara | a

            if nueva > objetivo:
                break

            grupo.append([a, I])
            bt(j + 1, nueva, grupo)
            grupo.pop()

    bt(0, 0, [])

    return resultados


def tamanos_de_grupos(grupos):
    """
    Retorna los tamaños distintos de los grupos, ordenados
    de menor a mayor.
    """

    return sorted(set(len(g) for g in grupos))

In [24]:
# Bajamos los factores del ideal a K y buscamos los productos que pertenecen a K.
def L1_factores(elem, L):

    I = K.fractional_ideal([elem])
    fac_K = I.factor()

    factors_K = []
    for f, e in fac_K:
        factors_K += [f] * e

    # Extendemos los factores a L, conservando sus multiplicidades.
    factors_L = []

    for J in factors_K:
        E = L.fractional_ideal(J.gens())
        factors_L.append(E)
        show(E)

    n = len(factors_L)
    fac_K = []

    # Recorremos los subconjuntos no vacíos de factores.
    for i in range(1, 2^n):
        r = i
        I = L.ideal([1])

        for j in range(n):
            if r >= 2^(n - 1 - j):
                I = I * factors_L[j]
                r -= 2^(n - 1 - j)

        # Verificamos si todos los generadores reducidos pertenecen a K.
        esta = True
        for g in I.gens_reduced():
            if g.minpoly().degree() != 1:
                esta = False

        if esta:
            fac_K.append([i, I])

    return grupos_irreducibles(fac_K, n)

In [25]:
# Bajamos los grupos de ideales de L1 a K.
def K_factores(elem, L):

    Facts = L1_factores(elem, L)
    Facts_en_K = []

    for grupo in Facts:
        todos_en_K = True

        for J in grupo:
            I = J.ideal_below()

            for g in I.gens_reduced():
                if g.minpoly().degree() != 1:
                    todos_en_K = False
                    break

            if not todos_en_K:
                break

        if todos_en_K:
            Facts_en_K.append(grupo)

    return Facts_en_K

In [26]:
# Calculamos los grupos de factores de elem que pertenecen a K.
elem = 30
Facts = K_factores(elem, L)

# Obtenemos las longitudes distintas de las factorizaciones.
Longitudes = tamanos_de_grupos(Facts)

Fractional ideal (-b)

Fractional ideal (-b)

Fractional ideal (((-1/10*a - 3/10)*b)*c)

Fractional ideal (((-1/10*a - 3/10)*b)*c)

Fractional ideal (-c)

Fractional ideal ((-1/5*a + 2/5)*c)

In [27]:
# Mostramos los grupos de ideales obtenidos.
for x in Facts:
    show(x)

[Fractional ideal (5), Fractional ideal (3), Fractional ideal (2)]

[Fractional ideal (5), Fractional ideal (6)]

[Fractional ideal (3), Fractional ideal (10)]

[Fractional ideal (15), Fractional ideal (2)]

[Fractional ideal (-a - 3), Fractional ideal (a - 3)]

[Fractional ideal (30)]